In [1]:
import itertools
import math

import cirq
import cirq_google
import numpy as np

# Google Quantum AI Challenge Problem

Your code may import from cirq, cirq_google, their dependencies, and python's standard library. Other imports are not allowed.

**Please upload your solutions to [this form](https://docs.google.com/forms/d/e/1FAIpQLScw93lwRb3OihfaHgpobs8TO6nXijwvkRVYjWPpNg7n92FjIA/viewform?usp=publish-editor).** See the form for further instructions.

Consider an $N$-qubit, weight $n$ Dicke state, defined by
$$|D^{(N)}_n\rangle = \begin{pmatrix} N \\ n \end{pmatrix}^{-1/2} \sum_k \mathcal{P}_k \left( |0\rangle^{\otimes(N-n)} \otimes |1\rangle^{\otimes n}\right),$$
where the sum is over all distinct permutations $\mathcal{P}_k$. For $n=1$, this is the $W$ state, given by
$$|W\rangle = \frac{1}{\sqrt N} \left( |100\ldots 0 \rangle + |010\ldots 0 \rangle + \ldots + |00\ldots 01 \rangle \right). $$

Your task is to complete the following function

# DuQuantum solution: exact, hardware-aware Dicke-state preparation on Willow

This section contains our complete solution. It follows the research plan in
`DuQuantum_Research_and_Implementation_Plan.md`: several *exact* constructions are generated,
placed on the calibrated device graph, compiled to the native gate set
(`CZ` + `PhasedXZGate`) with dynamical decoupling, and the candidate with the highest
*predicted* hardware success probability is returned.

**Notation.** $N$ = number of qubits, $n$ = number of excitations (the notebook's names),
$k=\min(n, N-n)$ = the excitation count we actually prepare (see the complement trick below).
$R_y(\theta)=e^{-i\theta Y/2}$, so $R_y(\theta)|0\rangle=\cos\frac\theta2|0\rangle+\sin\frac\theta2|1\rangle$.

**Pipeline.**

| Stage | What it does | Why |
| --- | --- | --- |
| 1. Exact building blocks | Splits, Givens rotations and "split-and-cyclic-shift" (SCS) blocks written directly in CZs | Every CZ costs ~0.25% error, so each block uses the fewest CZs we could prove or find |
| 2. Logical constructions | W-state tree ($k=1$), Bärtschi–Eidenbenz (BE) path and a two-arm "split" construction ($k\ge2$) | Different shapes trade CZ count against depth |
| 3. Placement | Search trees / paths / ladders on the calibrated coupler graph | Uses the best qubits, couplers and readout |
| 4. Compilation | Merge 1-qubit gates into `PhasedXZ`, ALAP layering into alternating 1q/CZ moments, XY dynamical decoupling | Native gate set, minimal idle exposure |
| 5. Selection | Predict $\log F$ (gates + readout + idling) for every candidate and keep the best | Calibration-aware choice |

**Complement trick.** $|D^{(N)}_n\rangle = X^{\otimes N}|D^{(N)}_{N-n}\rangle$ because flipping every bit maps
weight-$n$ strings bijectively onto weight-$(N-n)$ strings. We therefore always prepare
$k=\min(n,N-n)\le N/2$ excitations and, if $n>N/2$, append $X$ on every qubit. The extra $X$'s merge into
the final single-qubit layer, so the trick costs nothing.

In [2]:
import heapq  # priority queues for Prim-style tree growth and beam search
import math  # exact binomials, logs, arcsines
import time  # wall-clock budget (the challenge allows one minute per call)

import cirq  # circuit objects and native gates
import cirq_google  # Willow105 device description (used by the bonus solution)
import numpy as np  # 2x2 matrix algebra for gate merging

## 1. Exact building blocks

We build circuits from a tiny internal instruction list: `('U', q, M)` applies the $2\times2$ unitary `M`
to qubit `q`, and `('CZ', a, b)` applies a CZ. Single-qubit matrices are merged later into one
`PhasedXZGate` between consecutive CZs, so they are free in CZ count; only CZs matter.

**(a) CNOT from one CZ.** $\mathrm{CNOT}_{c\to t}=H_t\,\mathrm{CZ}_{ct}\,H_t$ because $HZH=X$.

**(b) Controlled rotation onto a known $|0\rangle$: one CZ.** For any $\varphi$,
$R_y(\varphi)\,Z\,R_y(-\varphi)=R_y(2\varphi)Z$ (since $ZR_y(\varphi)Z=R_y(-\varphi)$), and $Z|0\rangle=|0\rangle$. Hence
$$R_y(\tfrac\theta2)_t\,\mathrm{CZ}_{ct}\,R_y(-\tfrac\theta2)_t:\quad |0\rangle_c|0\rangle_t\mapsto|0\rangle|0\rangle,\qquad |1\rangle_c|0\rangle_t\mapsto|1\rangle R_y(\theta)|0\rangle .$$
A general controlled-$R_y$ needs two CZs; one suffices whenever the target is guaranteed to be
$|0\rangle$ when the control is $|1\rangle$. We use this "known-target" trick everywhere.

**(c) Excitation transfer (W-state split): two CZs.** $|1\rangle_s|0\rangle_d\mapsto\sqrt{1-p}\,|10\rangle+\sqrt p\,|01\rangle$,
$|00\rangle\mapsto|00\rangle$: a known-target rotation of $d$ followed by $\mathrm{CNOT}_{d\to s}$ (which clears $s$ when $d$
took the excitation). Two CZs are necessary: with one CZ, $(A_1\otimes B_1)\,\mathrm{CZ}\,(A_0\otimes B_0)$ applied to $|x\rangle|0\rangle$
gives $\sum_y (A_0)_{yx}A_1|y\rangle\otimes B_1Z^y|b\rangle$; keeping $|00\rangle$ a product state forces $A_0$ to be diagonal, and then
the $|10\rangle$ input also stays a product state, so the entangled branch cannot be produced.

**(d) Givens rotation: two CZs.** $G(\theta)=\exp\!\big[\tfrac\theta2(|10\rangle\langle01|-|01\rangle\langle10|)\big]=\exp\!\big[i\tfrac\theta4(XY-YX)\big]$
fixes $|00\rangle,|11\rangle$ and rotates $|01\rangle\mapsto\cos\frac\theta2|01\rangle+\sin\frac\theta2|10\rangle$. Conjugating by $S$ on qubit $b$ maps
$XY-YX\to-(XX+YY)$, and $R_x(\frac\pi2)^{\otimes2}$ maps $YY\to ZZ$. Finally
$\mathrm{CNOT}_{a\to b}\,[R_x(\alpha)\otimes R_z(\beta)]\,\mathrm{CNOT}_{a\to b}=e^{-i\alpha XX/2}e^{-i\beta ZZ/2}$, so
$$G(\theta)=V^\dagger W^\dagger\,\mathrm{CNOT}_{ab}\,[R_x(\tfrac\theta2)\otimes R_z(\tfrac\theta2)]\,\mathrm{CNOT}_{ab}\,W V,\quad V=S_b,\ W=R_x(\tfrac\pi2)^{\otimes2}.$$

**(e) Moving-hole gate $G_3$: five CZs** (an exhaustive numerical search found no 3- or 4-CZ circuit on a line). On qubits
$(c,a,b)$ with $a$ in the middle and input domain $\{000,001,011,101,111\}$ it maps $001\mapsto\cos\frac\theta2|001\rangle+\sin\frac\theta2|010\rangle$
(partial move of a "hole" from $a$ to $b$), $101\mapsto|110\rangle$ (full move when $c=1$, i.e. the hole is "in transit"), and fixes
the rest. Construction: $\mathrm{CNOT}_{a\to b}\,M\,\mathrm{CNOT}_{a\to b}$ with
$M=R_y(\alpha)Z^{c}R_y(\beta)Z^{b}R_y(-\beta)Z^{c}R_y(-\alpha)$ acting on $a$, $\alpha=\frac{\theta+\pi}4$, $\beta=\frac{\theta-\pi}4$. After the first
CNOT, $b=1\Rightarrow a=0$, and pushing the $Z$'s through the rotations gives total $R_y$ angles $0$ ($c=b=0$), $0$ ($c=1,b=0$),
$\theta$ ($c=0,b=1$) and $\pi$ ($c=b=1$), exactly as required.

**(f) Merged top block: four CZs instead of 2 + 5 = 7.** Found by an exhaustive numerical search over CZ orderings
(only the ordering $ab,bc,ab,bc$ works) and then derived analytically. On a line $a-b-c$ with domain
$\{000,001,011\}$:
1. known-target rotation $b\to a$ with angle $\theta_2$ (only the $011$ input has $b=1$, and there $a=0$);
2. a Givens rotation $G(\theta_1)$ on $(b,c)$ whose middle $R_x(\theta_1/2)$ on $b$ is replaced by
   $R_x(\frac\pi2)\,\mathrm{CZ}_{ab}\,R_x(\frac{\theta_1-\pi}2)$. For $a=0$ nothing changes (the $R_x$ angles add to $\theta_1/2$), so
   $001\mapsto\cos\frac{\theta_1}2|001\rangle+\sin\frac{\theta_1}2|010\rangle$ and $011$ is fixed. For $a=1$ the middle becomes $R_x(\pi-\frac{\theta_1}2)Z$;
   moving the $Z$ out of the Givens frame shows that the $|11\rangle_{bc}$ input is rotated by an extra $\pi-\theta_1$ in the
   $\{|01\rangle,|10\rangle\}$ plane and lands exactly on $|10\rangle_{bc}$. So $|111\rangle\mapsto|110\rangle$ up to a phase $-i$;
3. an $S$ gate on $a$ removes that phase ($a=1$ only in that branch).

Every block is checked numerically against its defining isometry in the verification section.

In [3]:
_I2 = np.eye(2, dtype=complex)  # identity
_X = np.array([[0, 1], [1, 0]], dtype=complex)  # Pauli X
_Y = np.array([[0, -1j], [1j, 0]], dtype=complex)  # Pauli Y
_H = np.array([[1, 1], [1, -1]], dtype=complex) / np.sqrt(2)  # Hadamard
_S = np.diag([1, 1j]).astype(complex)  # phase gate S = sqrt(Z)


def _RY(theta):
    """R_y(theta) = exp(-i theta Y / 2): real rotation, |0> -> cos(theta/2)|0> + sin(theta/2)|1>."""
    c, s = math.cos(theta / 2), math.sin(theta / 2)  # half-angle cosine and sine
    return np.array([[c, -s], [s, c]], dtype=complex)  # standard real rotation matrix


def _RX(theta):
    """R_x(theta) = exp(-i theta X / 2)."""
    c, s = math.cos(theta / 2), math.sin(theta / 2)  # half-angle cosine and sine
    return np.array([[c, -1j * s], [-1j * s, c]], dtype=complex)  # rotation about the X axis


def _RZ(theta):
    """R_z(theta) = exp(-i theta Z / 2)."""
    return np.diag([np.exp(-0.5j * theta), np.exp(0.5j * theta)])  # diagonal phase rotation


def _u(q, m):
    """Instruction: apply the 2x2 unitary m to qubit q."""
    return ('U', q, m)  # tuples are cheap; matrices are merged later


def _cz(a, b):
    """Instruction: CZ between qubits a and b (symmetric)."""
    return ('CZ', a, b)  # the only two-qubit gate we ever emit


def _cnot(control, target):
    """CNOT = H_t CZ H_t (one CZ, derivation (a))."""
    return [_u(target, _H), _cz(control, target), _u(target, _H)]  # HZH = X on the target


def _cry_on_zero(control, target, theta):
    """Controlled-R_y(theta), valid when target = |0> whenever control = |1> (one CZ, derivation (b))."""
    return [_u(target, _RY(-theta / 2)), _cz(control, target), _u(target, _RY(theta / 2))]  # R_y(t/2) CZ R_y(-t/2)


def _split_angle(p):
    """Angle theta with sin^2(theta/2) = p, i.e. R_y(theta)|0> has |1>-probability p."""
    return 2 * math.asin(math.sqrt(min(max(p, 0.0), 1.0)))  # clamping guards tiny float overshoots


def _transfer(src, dst, p):
    """|1>_src|0>_dst -> sqrt(1-p)|10> + sqrt(p)|01>, |00> -> |00> (two CZs, derivation (c))."""
    return _cry_on_zero(src, dst, _split_angle(p)) + _cnot(dst, src)  # rotate dst, then clear src if dst took it


def _givens(a, b, theta, middle_a=None):
    """Givens rotation G(theta) on (a, b) with two CZs (derivation (d)).

    `middle_a` optionally replaces the middle R_x(theta/2) on qubit a (used by the merged top block).
    """
    if middle_a is None:  # plain Givens rotation
        middle_a = [_u(a, _RX(theta / 2))]  # the XX-part angle
    ops = [_u(b, _S), _u(a, _RX(np.pi / 2)), _u(b, _RX(np.pi / 2))]  # W V: map XY-YX onto -(XX+ZZ)
    ops += _cnot(a, b)  # CNOT conjugation turns X_a into X_a X_b and Z_b into Z_a Z_b ...
    ops += middle_a + [_u(b, _RZ(theta / 2))]  # ... so these rotations become exp(-i t XX/2) exp(-i t ZZ/2)
    ops += _cnot(a, b)  # close the CNOT conjugation
    ops += [_u(a, _RX(-np.pi / 2)), _u(b, _RX(-np.pi / 2)), _u(b, _S.conj().T)]  # undo W and V
    return ops


def _g3(c, a, b, theta):
    """Moving-hole gate on (c, a, b): partial move a->b if c=0, full move if c=1 (five CZs, derivation (e))."""
    al = (theta + np.pi) / 4  # alpha in the derivation
    be = (theta - np.pi) / 4  # beta in the derivation
    ops = _cnot(a, b)  # afterwards b = 1 implies a = 0, enabling known-target rotations of a
    ops += [_u(a, _RY(-al)), _cz(c, a), _u(a, _RY(-be)), _cz(b, a), _u(a, _RY(be)), _cz(c, a), _u(a, _RY(al))]  # M
    ops += _cnot(a, b)  # map |a=1,b=1> back to |a=1,b=0>, completing the excitation move
    return ops


def _top_block(a, b, c, p_top, p_next):
    """Merged first two steps of an SCS sweep on a line a-b-c, four CZs (derivation (f)).

    Domain {000, 001, 011} (qubit order a, b, c). The excitation on c moves to b with probability p_next
    when the input is 001, and to a with probability p_top when the input is 011.
    """
    t2 = _split_angle(p_top)  # rotation for the 011 branch (landing site a)
    t1 = _split_angle(p_next)  # rotation for the 001 branch (landing site b)
    ops = _cry_on_zero(b, a, t2)  # step 1: only 011 has b = 1, and there a = 0
    middle = [_u(b, _RX((t1 - np.pi) / 2)), _cz(a, b), _u(b, _RX(np.pi / 2))]  # step 2: Z_b inserted iff a = 1
    ops += _givens(b, c, t1, middle_a=middle)  # Givens on (b, c) with the modified middle layer
    ops += [_u(a, _S)]  # step 3: cancel the -i phase acquired by the a = 1 branch
    return ops

## 2. Logical constructions

### 2.1 W states ($k=1$): excitation tree, $2N-3$ CZs, depth $O(\log N)$

Pick a spanning tree of $N$ qubits with root $r$, and write $s(v)$ for the size of the subtree under $v$. Starting from
$|1\rangle_r$, every node $v$ that currently "owns" the excitation for $m_v$ qubits (itself plus its not-yet-served child subtrees)
transfers it to child $c$ with probability $s(c)/m_v$. The probability that the excitation reaches a given qubit is the product
of conditional probabilities along its root path, which telescopes to $1/N$; all amplitudes stay real and positive, so the
result is exactly $|W_N\rangle$.

The *first* transfer from the root acts on the product state $|1\rangle_r|0\rangle_c$, so it needs only one CZ:
prepare $r$ in $\sqrt p|0\rangle+\sqrt{1-p}|1\rangle$, then apply $\mathrm{CNOT}_{r\to c}$ and $X_c$. Total: $1+2(N-2)=2N-3$ CZs. An exhaustive numerical
search confirmed that $W_3$ needs 3 and $W_4$ needs 5 CZs, so this count is optimal at those sizes.

Children are served in order of decreasing subtree height, so the longest chain starts first; the depth is roughly
$2\times$(tree height + branching), i.e. logarithmic in $N$ for a bushy tree on the 2D grid.

### 2.2 General $k$: Bärtschi–Eidenbenz (BE) on a path

BE prepares $|D^{(N)}_k\rangle=U_N|0^{N-k}1^k\rangle$ with $U_N=\prod_{m=N}^{2}\mathrm{SCS}_m$. Before step $m$ the first $m$ positions hold
$|0^{m-r}1^r\rangle$ (ones right-justified, $r$ = remaining excitations). Since
$$|D^{(m)}_r\rangle=\sqrt{\tfrac rm}\,|D^{(m-1)}_{r-1}\rangle|1\rangle+\sqrt{\tfrac{m-r}m}\,|D^{(m-1)}_r\rangle|0\rangle,$$
step $m$ must map $|0^{m-r}1^r\rangle\mapsto\sqrt{\tfrac rm}|0^{m-r}1^r\rangle+\sqrt{\tfrac{m-r}m}|0^{m-r-1}1^r0\rangle$: either position $m$ keeps its 1, or the
"hole" at the boundary $m-r$ travels to $m$ (a cyclic shift). Only the window $m-k,\dots,m$ is touched.

We implement $\mathrm{SCS}_m$ as a *local sweep* along the window, so it needs only nearest-neighbour CZs: the hole moves
one site at a time, partially at its original boundary and fully while "in transit". The first two sweep steps form the
4-CZ merged top block; the remaining steps are $G_3$ gates. Cost per step: 2 CZs ($k=1$), 4 CZs ($k=2$), $4+5(k-2)$ in general
(textbook BE uses one non-local three-qubit gate per extra branch, each several CNOTs, **plus** SWAP routing on a line).

*Caveat handled in code:* when $m\le k$ the window is the whole prefix and the all-ones input $|1^m\rangle$ can occur; the
merged block is not valid there (numerically it would need at least 6 CZs), so those steps use the plain sweep, which fixes $|1\dots1\rangle$.

### 2.3 General $k$: two-arm "split" construction (our main $k\ge2$ workhorse)

Split the qubits into a left arm $L$ ($a$ qubits) and a right arm $R$ ($b=N-a$ qubits). Then
$$|D^{(N)}_k\rangle=\sum_j\sqrt{P(j)}\;|D^{(a)}_{k-j}\rangle_L|D^{(b)}_j\rangle_R,\qquad P(j)=\frac{\binom bj\binom a{k-j}}{\binom Nk}\ \text{(hypergeometric)}.$$
Because the total weight $k$ is *fixed* at the start, the weight distribution is cheap to prepare in unary:
1. **Thermometer chain** on the first $k$ qubits $r_1..r_k$ of $R$: $r_1\leftarrow R_y$ with $P(j\ge1)$, then known-target rotations
   $r_{u-1}\to r_u$ with conditional probabilities $P(j\ge u)/P(j\ge u-1)$. Result: $\sum_j\sqrt{P(j)}|1^j0^{k-j}\rangle$ ($k-1$ CZs, exact).
2. **Rung copy.** The left arm's first $k$ qubits are laid out as a parallel row in *reverse* order, so that $l_{k+1-u}$ neighbours
   $r_u$ (a "ladder"). The left arm needs the thermometer $l_v=[k-j\ge v]=\lnot r_{k+1-v}$, so one parallel layer of CNOTs plus $X$ gates
   writes it ($k$ CZs, depth 1).
3. Each arm now holds a right-justified unary input, so we run the BE unitary on both arms **in parallel**.

Compared with one long BE path, the two arms run concurrently (about half the depth) and the construction also *saves CZs*,
because the expensive full-window sweeps happen on half as many positions. Examples: $(N,k)=(6,3)$: 23 vs 36 CZs; $(8,4)$: 49 vs 77.

In [4]:
def _scs_ops(window, m, all_ones_possible):
    """SCS_m on window = [p_k, ..., p_1, p_0] (p_0 = position m) as a nearest-neighbour sweep."""
    k = len(window) - 1  # window size minus one = largest number of excitations handled
    if k == 1:  # a single partial move = Givens rotation (it also fixes |11>)
        return _givens(window[0], window[1], _split_angle((m - 1) / m))  # move the hole with probability (m-1)/m
    ops = []  # accumulated instructions
    i = k  # current hole index: branch r = i has its hole at p_i
    if not all_ones_possible:  # the merged top block needs p_k = 0 on every input
        ops += _top_block(window[0], window[1], window[2], (m - k) / m, (m - k + 1) / m)  # branches r = k and k-1
        i = k - 2  # the block already performed the top two sweep steps
    while i >= 1:  # remaining sweep steps, top to bottom
        theta = _split_angle((m - i) / m)  # branch r = i moves its hole with probability (m-i)/m
        if i == k:  # very first step of an unmerged sweep: no "in transit" control yet
            ops += _givens(window[0], window[1], theta)  # partial move p_k <-> p_{k-1}
        else:  # generic step: the control is the site above the hole position
            ops += _g3(window[k - i - 1], window[k - i], window[k - i + 1], theta)  # (c, a, b) = (p_{i+1}, p_i, p_{i-1})
        i -= 1  # move one site towards the head p_0
    return ops


def dicke_unitary_ops(path, k):
    """BE unitary on `path`: maps |0^{N-r} 1^r> (ones at the END of the list) to |D^(N)_r> for every r <= k."""
    N = len(path)  # number of positions
    k = min(k, N)  # cannot hold more excitations than qubits
    ops = []  # accumulated instructions
    for m in range(N, 1, -1):  # decide positions N, N-1, ..., 2 (position 1 is then determined)
        kk = min(k, m - 1)  # the window reaches back k sites, but not past position 1
        ops += _scs_ops(list(path[m - 1 - kk: m]), m, all_ones_possible=(k >= m))  # all-ones input reachable iff k >= m
    return ops


def dicke_path_ops(path, k):
    """|D^(N)_k> on a path: X on the last k sites, then the BE unitary."""
    return [_u(q, _X) for q in path[len(path) - k:]] + dicke_unitary_ops(path, k)  # |0^{N-k}1^k>, then U_N


def split_ops(left, right, k):
    """Two-arm construction (Section 2.3).

    left = [l_1, ..., l_a], right = [r_1, ..., r_b]; l_1 and r_1 are the arm heads.
    Ladder requirement: l_{k+1-u} must neighbour r_u for every rung that is actually used.
    """
    a, b = len(left), len(right)  # arm lengths
    N = a + b  # total number of qubits
    P = [math.comb(b, j) * math.comb(a, k - j) / math.comb(N, k) if (0 <= k - j <= a and j <= b) else 0.0
         for j in range(k + 1)]  # hypergeometric law of j = number of excitations in the right arm
    tail = [sum(P[u:]) for u in range(k + 2)]  # tail[u] = P(j >= u)
    ops = []  # accumulated instructions
    for u in range(1, min(k, b) + 1):  # thermometer chain r_1 ... r_k
        if tail[u] <= 1e-15:  # j >= u impossible: r_u stays |0>, and so do all later r's
            break  # nothing more to prepare
        if u == 1:  # first bit from scratch
            ops.append(_u(right[0], _RY(_split_angle(tail[1]))))  # P(r_1 = 1) = P(j >= 1)
        else:  # later bits conditioned on the previous one (known |0> target)
            ops += _cry_on_zero(right[u - 2], right[u - 1], _split_angle(tail[u] / tail[u - 1]))  # P(j>=u | j>=u-1)
    for u in range(1, k + 1):  # rung copies l_{k+1-u} = NOT r_u
        v = k + 1 - u  # left index paired with r_u
        if v > a:  # left arm too short: this left bit does not exist (then r_u is always 1)
            continue  # nothing to write
        if u > b or tail[u] <= 1e-15:  # r_u is always 0, so l_v is always 1
            ops.append(_u(left[v - 1], _X))  # deterministic bit, no CZ needed
        elif tail[u] < 1 - 1e-15:  # r_u genuinely random: copy its complement across the rung
            ops += _cnot(right[u - 1], left[v - 1]) + [_u(left[v - 1], _X)]  # l_v = 1 XOR r_u
        # otherwise r_u is always 1 and l_v correctly stays |0>
    ops += dicke_unitary_ops(list(reversed(right)), min(k, b))  # BE on the right arm (head r_1 = last element)
    ops += dicke_unitary_ops(list(reversed(left)), min(k, a))  # BE on the left arm; runs in parallel
    return ops


def _subtree_sizes(root, children):
    """Subtree sizes s(v) for a rooted tree given as {node: [children]}."""
    order = [root]  # BFS order
    for v in order:  # the list grows while we iterate
        order.extend(children.get(v, []))  # append children
    size = {}  # node -> subtree size
    for v in reversed(order):  # leaves first
        size[v] = 1 + sum(size[c] for c in children.get(v, []))  # itself plus its children's subtrees
    return size


def w_tree_ops(root, children):
    """|W_N> on a rooted tree with 2N-3 CZs (Section 2.1)."""
    size = _subtree_sizes(root, children)  # s(v)
    kids = children.get(root, [])  # root's children in serving order
    if not kids:  # single qubit: |W_1> = |1>
        return [_u(root, _X)]  # just flip it
    first = kids[0]  # first child, served with the 1-CZ product-state split
    p = size[first] / size[root]  # probability that the excitation goes to the first child's subtree
    ops = [_u(root, _RY(_split_angle(1 - p)))]  # root = sqrt(p)|0> + sqrt(1-p)|1>
    ops += _cnot(root, first) + [_u(first, _X)]  # -> sqrt(p)|01> + sqrt(1-p)|10>
    stack = [root]  # depth-first traversal keeps every qubit's instruction order valid
    while stack:  # visit every node once
        v = stack.pop()  # current owner of an excitation
        remaining = size[v] - (size[first] if v == root else 0)  # mass still owned by v
        for c in children.get(v, []):  # serve the children in order
            if v == root and c == first:  # already served by the 1-CZ split
                continue  # skip it
            ops += _transfer(v, c, size[c] / remaining)  # hand child c its share
            remaining -= size[c]  # v now owns less
        stack.extend(reversed(children.get(v, [])))  # children then process their own subtrees
    return ops

## 3. Compilation: native gates, layering, dynamical decoupling

1. **Merge.** All single-qubit matrices on a qubit between two of its CZs are multiplied into one unitary and emitted
   as a single `PhasedXZGate` (every single-qubit unitary is a `PhasedXZ` up to global phase). This minimises the
   single-qubit gate count and error.
2. **Layer.** CZs form a DAG (two CZs depend on each other if they share a qubit). We compute the ASAP depth $D$ and then
   place every CZ **as late as possible** (ALAP) within $D$ layers. Moments alternate *1q layer / CZ layer*, which matches
   how Google calibrates parallel CZ moments. ALAP keeps each qubit in $|0\rangle$ for as long as possible, and $|0\rangle$ is
   immune to both $T_1$ decay and dephasing, so idle error only accumulates after a qubit's first gate.
3. **Dynamical decoupling (DD).** Once a qubit is active, every 1q slot in which it would otherwise idle receives an $X$ or $Y$
   pulse (alternating, XY4-style). The last slot of each idle window receives the inverse of the accumulated Pauli product,
   merged with the real gate that belongs there. Paulis multiply to a Pauli and the window contains no CZ on that qubit, so
   the logical unitary is **exactly** unchanged, while quasi-static $Z$ errors (detuning, spectator phases from neighbouring CZs)
   are echoed out. The challenge text flags DD as important; our emulator (later section) quantifies the trade-off, since each
   pulse costs about one single-qubit error.

In [5]:
def _is_identity(u, atol=1e-9):
    """True if u is the identity up to a global phase."""
    phase = u[0, 0] if abs(u[0, 0]) > 0.5 else 1.0  # an identity-like matrix has |u00| ~ 1
    return np.allclose(u / phase, _I2, atol=atol)  # compare after removing the phase


def _phxz(q, u):
    """Native gate for an arbitrary single-qubit unitary."""
    return cirq.PhasedXZGate.from_matrix(u).on(q)  # exact up to a global phase


def compile_layers(ops, dd=True):
    """Instruction list -> cirq.Circuit with alternating PhasedXZ / CZ moments (ALAP, optional DD)."""
    pending = {}  # qubit -> product of 1q matrices since its last CZ
    last = {}  # qubit -> index of its most recent CZ node
    nodes = []  # CZ nodes: qubits, pre-gates, DAG edges
    for kind, a, b in ops:  # walk the instructions in program order
        if kind == 'U':  # single-qubit matrix b on qubit a
            pending[a] = b @ pending.get(a, _I2)  # left-multiply: later gates act after earlier ones
            continue  # nothing else to do
        nodes.append({'qubits': (a, b),  # a CZ node
                      'pre': {a: pending.pop(a, None), b: pending.pop(b, None)},  # 1q gates right before it
                      'preds': {last[q] for q in (a, b) if q in last},  # CZs that must come first
                      'succs': set()})  # filled below
        last[a] = last[b] = len(nodes) - 1  # this CZ is now the latest on both qubits
    trailing = pending  # 1q gates after a qubit's last CZ (or the only gates of CZ-free qubits)
    for i, nd in enumerate(nodes):  # build successor lists
        for p in nd['preds']:  # every predecessor ...
            nodes[p]['succs'].add(i)  # ... points forward to i
    asap = [0] * len(nodes)  # earliest CZ layer (1-based)
    for i, nd in enumerate(nodes):  # program order is a topological order
        asap[i] = 1 + max((asap[p] for p in nd['preds']), default=0)  # one after the latest predecessor
    depth = max(asap, default=0)  # number of CZ layers
    layer = [0] * len(nodes)  # ALAP layer
    for i in range(len(nodes) - 1, -1, -1):  # reverse topological order
        layer[i] = min((layer[s] for s in nodes[i]['succs']), default=depth + 1) - 1  # one before the earliest successor
    n_mom = 2 * depth + 1  # moments: 1q, CZ, 1q, CZ, ..., 1q
    slots = [dict() for _ in range(n_mom)]  # 1q moments: qubit -> matrix
    cz_moments = [[] for _ in range(n_mom)]  # CZ moments
    q_layers = {}  # qubit -> list of (CZ layer, pre-gate)
    for i, nd in enumerate(nodes):  # place the CZs
        cz_moments[2 * layer[i] - 1].append(cirq.CZ(*nd['qubits']))  # CZ layer t lives in moment 2t-1
        for q in nd['qubits']:  # remember each qubit's timeline
            q_layers.setdefault(q, []).append((layer[i], nd['pre'][q]))  # its pre-gate goes right before this CZ
    for q in set(q_layers) | set(trailing):  # every qubit that appears
        seq = sorted(q_layers.get(q, []), key=lambda t: t[0])  # its CZs in time order
        if not seq:  # no CZ at all: put its only gate in the final 1q moment (stay |0> as long as possible)
            u = trailing.get(q)  # its merged gate
            if u is not None and not _is_identity(u):  # skip identities
                slots[n_mom - 1][q] = u  # last moment
            continue  # done with this qubit
        t0, u0 = seq[0]  # first CZ of this qubit
        if u0 is not None:  # gates before the first CZ (the qubit is |0> before them, so no DD there)
            slots[2 * t0 - 2][q] = u0  # 1q moment right before CZ layer t0
        windows = [(seq[j][0], seq[j + 1][0], seq[j + 1][1], 'mid') for j in range(len(seq) - 1)]  # between CZs
        windows.append((seq[-1][0], depth + 1, trailing.get(q), 'tail'))  # after the last CZ, until the end
        for t_a, t_b, u, kind in windows:  # each idle window of an active qubit
            moms = [2 * t for t in range(t_a, t_b)]  # its 1q moments
            u = _I2 if u is None else u  # net single-qubit operation required in this window
            if not dd or len(moms) < 2:  # no idle CZ layer inside (or DD disabled)
                if not _is_identity(u):  # only emit real gates
                    slots[moms[-1] if kind == 'mid' else moms[0]][q] = u  # just before next CZ / just after last CZ
                continue  # next window
            pulses = [(_X if i % 2 == 0 else _Y) for i in range(len(moms) - 1)]  # X, Y, X, Y, ...
            acc = _I2  # running product of the pulses
            for p in pulses:  # accumulate
                acc = p @ acc  # later pulses act after earlier ones
            pulses.append(acc.conj().T)  # the final slot undoes the product: window total = identity
            if kind == 'mid':  # the real gate must act right before the next CZ
                pulses[-1] = u @ pulses[-1]  # merge it into the final slot
            else:  # tail window: apply the real gate first, then decouple
                pulses[0] = pulses[0] @ u  # merge it into the first slot
            for mom, p in zip(moms, pulses):  # emit
                if not _is_identity(p):  # identity slots stay empty
                    slots[mom][q] = p  # one PhasedXZ per qubit per moment
    moments = []  # assemble the circuit
    for i in range(n_mom):  # in time order
        ops_i = [_phxz(q, u) for q, u in slots[i].items()] if i % 2 == 0 else cz_moments[i]  # 1q or CZ moment
        if ops_i:  # drop empty moments
            moments.append(cirq.Moment(ops_i))  # one homogeneous moment
    return cirq.Circuit(moments)  # final native circuit

## 4. Calibration model, success-probability estimate, placement search

**Pessimistic error rates.** Each calibration entry is a measured mean $\hat e$ with statistical uncertainty $\sigma$. Choosing the
minimum of many noisy estimates systematically selects under-estimates (the "winner's curse"), so we rank with
$e=\hat e+\beta\sigma$, $\beta=1$ (a one-standard-error penalty), following the plan's uncertainty-aware placement.

**Predicted log-fidelity** of a compiled candidate (higher is better):
$$\log F=\sum_{\mathrm{CZ}}\log(1-e_{\mathrm{CZ}})+\sum_{\mathrm{1q}}\log(1-e_{\mathrm{1q}})+\sum_{q}\log\!\big(1-[(1-p)e^{0\to1}_q+p\,e^{1\to0}_q]\big)-\sum_q\frac{t^{\mathrm{active}}_q}{T_{\mathrm{idle}}}.$$
The readout term uses the exact excitation probability $p=n/N$ of a Dicke qubit (asymmetric readout matters: $e^{1\to0}\approx2e^{0\to1}$).
$t^{\mathrm{active}}_q$ is the time from the qubit's first gate to measurement (25 ns per 1q moment, 42 ns per CZ moment), and
$T_{\mathrm{idle}}=60\,\mu$s is a nominal combined $T_1$/dephasing time for Willow-class qubits with DD (it only has to rank
candidates, not predict absolute numbers).

**Search spaces** (exhaustive over starting points, pruned by cheap proxies, then compiled and scored exactly):
* *Trees* ($k=1$): for every root and three depth penalties, a Prim-like greedy growth adds the cheapest frontier qubit
  (2 CZs on its edge + readout + single-qubit cost + depth penalty).
* *Paths* (BE): beam search over simple paths, with position-dependent weights equal to the actual number of CZs / 1q gates
  the construction places on each edge / qubit.
* *Ladders* (split): every straight $2\times k$ ladder on the grid, extended into two disjoint arms by beam search.

In [6]:
T_1Q = 25e-9  # duration of a single-qubit moment (s)
T_CZ = 42e-9  # duration of a CZ moment (s)
T_IDLE = 60e-6  # nominal effective coherence time used to price idling (s)


class Calibration:
    """Pessimistic error rates and the usable coupling graph built from the four calibration dictionaries."""

    def __init__(self, cz, sq, e0, e1, beta=1.0):
        def pess(v):  # mean + beta * sigma, clipped to a sane range
            mean, std = v  # calibration tuple
            return float(min(max(mean + beta * std, 1e-6), 0.5))  # keeps the logarithms finite

        self.sq = {q: pess(v) for q, v in sq.items()}  # usable qubits = those with single-qubit data
        med0 = float(np.median([v[0] for v in e0.values()])) if e0 else 0.01  # fallback 0->1 readout error
        med1 = float(np.median([v[0] for v in e1.values()])) if e1 else 0.02  # fallback 1->0 readout error
        self.e0 = {q: pess(e0[q]) if q in e0 else med0 for q in self.sq}  # 0 -> 1 readout error
        self.e1 = {q: pess(e1[q]) if q in e1 else med1 for q in self.sq}  # 1 -> 0 readout error
        self.cz = {}  # frozenset pair -> CZ error
        for pair, v in cz.items():  # only couplers whose two qubits are usable
            a, b = tuple(pair)  # unpack (tuple or frozenset)
            if a in self.sq and b in self.sq and a != b:  # both ends usable
                key = frozenset((a, b))  # orientation-free key
                self.cz[key] = min(self.cz.get(key, 1.0), pess(v))  # keep the better value if listed twice
        self.adj = {q: set() for q in self.sq}  # adjacency of the usable graph
        for key in self.cz:  # fill it
            a, b = tuple(key)  # endpoints
            self.adj[a].add(b)  # undirected ...
            self.adj[b].add(a)  # ... edge

    def edge_cost(self, a, b):
        """-log(1 - CZ error) for coupler (a, b)."""
        return -math.log(1 - self.cz[frozenset((a, b))])  # additive cost

    def ro_cost(self, q, p):
        """-log(1 - readout error) for a qubit that is |1> with probability p."""
        return -math.log(1 - ((1 - p) * self.e0[q] + p * self.e1[q]))  # mixture of both error types

    def sq_cost(self, q):
        """-log(1 - single-qubit error)."""
        return -math.log(1 - self.sq[q])  # additive cost


def estimate_log_fidelity(circuit, cal, data_qubits, p_exc):
    """Predicted log success probability of a compiled circuit (see the formula above)."""
    total = 0.0  # running log-fidelity
    first = {}  # qubit -> index of its first moment
    durations = []  # moment durations
    for i, moment in enumerate(circuit):  # scan the moments
        is_cz = any(len(op.qubits) == 2 for op in moment)  # moments are homogeneous: CZ or 1q
        durations.append(T_CZ if is_cz else T_1Q)  # its duration
        for op in moment:  # gate errors
            for q in op.qubits:  # remember when each qubit becomes active
                first.setdefault(q, i)  # first occurrence only
            if len(op.qubits) == 2:  # CZ
                total += math.log(1 - cal.cz[frozenset(op.qubits)])  # its error
            else:  # PhasedXZ
                total += math.log(1 - cal.sq[op.qubits[0]])  # its error
    cum = np.cumsum([0.0] + durations)  # cumulative time at each moment boundary
    for q, i in first.items():  # idling from the first gate until measurement
        total -= (cum[-1] - cum[i]) / T_IDLE  # linearised decay
    for q in data_qubits:  # readout
        total -= cal.ro_cost(q, p_exc)  # its error
    return total


def _tree_children(parent):
    """{child: parent} -> {node: [children sorted by decreasing subtree height]}."""
    children = {}  # node -> children
    for v, u in parent.items():  # invert the parent map
        children.setdefault(u, []).append(v)  # u is v's parent
    height = {}  # memoised subtree heights

    def h(v):  # height of v's subtree
        if v not in height:  # compute once
            height[v] = 1 + max((h(c) for c in children.get(v, [])), default=0)  # 1 + tallest child
        return height[v]  # cached value

    for v in list(children):  # order every child list
        children[v].sort(key=lambda c: (-h(c), str(c)))  # tallest first (critical path first), deterministic ties
    return children


def grow_tree(cal, root, N, p_exc, depth_weight):
    """Prim-like greedy N-node tree from `root`; returns {child: parent} or None."""
    in_tree = {root}  # nodes already in the tree
    parent = {}  # child -> parent
    depth = {root: 0}  # tree depth of each node
    heap = []  # frontier edges keyed by cost

    def push(u):  # add u's frontier edges
        for v in cal.adj[u]:  # neighbours
            if v not in in_tree:  # only new nodes
                c = (2 * cal.edge_cost(u, v) + cal.ro_cost(v, p_exc) + 4 * cal.sq_cost(v)
                     + depth_weight * (depth[u] + 1))  # 2 CZs + readout + ~4 1q gates + depth penalty
                heapq.heappush(heap, (c, str(v), v, u))  # str(v) breaks ties deterministically

    push(root)  # start from the root
    while len(in_tree) < N and heap:  # grow until N nodes
        _, _, v, u = heapq.heappop(heap)  # cheapest frontier edge
        if v in in_tree:  # stale entry
            continue  # skip
        in_tree.add(v)  # accept v
        parent[v] = u  # reached via u
        depth[v] = depth[u] + 1  # one level deeper
        push(v)  # extend the frontier
    return parent if len(in_tree) == N else None  # None if the component is too small


def beam_paths(cal, N, edge_w, node_w, p_exc, beam=200):
    """Beam search for low-cost simple paths of N qubits with position-dependent weights."""
    beams = [(node_w[0] * cal.sq_cost(q) + cal.ro_cost(q, p_exc), (q,)) for q in sorted(cal.sq, key=str)]  # all starts
    for pos in range(1, N):  # append one position at a time
        cand = {}  # dedupe partial paths by (qubit set, end)
        for cost, path in beams:  # expand every beam
            u = path[-1]  # current end
            for v in cal.adj[u]:  # possible next qubit
                if v in path:  # simple paths only
                    continue  # skip
                c = cost + edge_w[pos - 1] * cal.edge_cost(u, v) + node_w[pos] * cal.sq_cost(v) + cal.ro_cost(v, p_exc)  # new cost
                key = (frozenset(path) | {v}, v)  # equivalent partial paths
                if key not in cand or cand[key][0] > c:  # keep the best representative
                    cand[key] = (c, path + (v,))  # extended path
        beams = heapq.nsmallest(beam, cand.values(), key=lambda t: t[0])  # prune
        if not beams:  # dead end everywhere
            return []  # no path of this length
    return beams  # sorted best-first


def extend_path(cal, prefix, length, forbidden, p_exc, w_edge, w_node, beam=30):
    """Beam-search continuations of `prefix` to `length` qubits avoiding `forbidden`."""
    beams = [(0.0, tuple(prefix))]  # start from the prefix
    while len(beams[0][1]) < length:  # until long enough
        cand = {}  # dedupe
        for cost, path in beams:  # expand
            u = path[-1]  # current end
            for v in cal.adj[u]:  # neighbours
                if v in path or v in forbidden:  # keep the arms disjoint and simple
                    continue  # skip
                c = cost + w_edge * cal.edge_cost(u, v) + w_node * cal.sq_cost(v) + cal.ro_cost(v, p_exc)  # new cost
                key = (frozenset(path) | {v}, v)  # equivalence class
                if key not in cand or cand[key][0] > c:  # keep the best
                    cand[key] = (c, path + (v,))  # extended path
        beams = heapq.nsmallest(beam, cand.values(), key=lambda t: t[0])  # prune
        if not beams:  # cannot extend
            return []  # failure
    return beams  # best-first continuations


def ladder_candidates(cal, k, p_exc, w_edge, w_node):
    """All straight 2 x k ladders: right row r_1..r_k and a left row with l_{k+1-u} next to r_u."""
    out = []  # (cost, left_head, right_head)
    for q in sorted(cal.sq, key=str):  # r_1
        for d in ((0, 1), (0, -1), (1, 0), (-1, 0)):  # direction of the right row
            for s in ((d[1], d[0]), (-d[1], -d[0])):  # side on which the left row lies
                right = [cirq.GridQubit(q.row + u * d[0], q.col + u * d[1]) for u in range(k)]  # r_1..r_k
                left = [cirq.GridQubit(r.row + s[0], r.col + s[1]) for r in reversed(right)]  # l_1..l_k (reversed)
                if any(x not in cal.sq for x in right + left):  # all qubits must be usable
                    continue  # skip
                edges = (list(zip(right, right[1:])) + list(zip(left, left[1:]))
                         + [(right[u], left[k - 1 - u]) for u in range(k)])  # rows + rungs
                if any(frozenset(e) not in cal.cz for e in edges):  # all couplers must be usable
                    continue  # skip
                cost = (sum(w_edge * cal.edge_cost(*e) for e in edges)
                        + sum(w_node * cal.sq_cost(x) + cal.ro_cost(x, p_exc) for x in right + left))  # proxy cost
                out.append((cost, left, right))  # keep
    out.sort(key=lambda t: t[0])  # cheapest first
    return out


def path_profile(N, k):
    """CZ count per edge and gate count per position of the BE construction (weights for the path search)."""
    qs = cirq.LineQubit.range(N)  # abstract line
    circ = compile_layers(dicke_path_ops(qs, k), dd=False)  # compile once
    pos = {q: i for i, q in enumerate(qs)}  # qubit -> position index
    edge_w = [0] * max(N - 1, 1)  # CZs per edge
    node_w = [0] * N  # 1q gates per position
    for op in circ.all_operations():  # tally
        if len(op.qubits) == 2:  # CZ (always nearest-neighbour)
            edge_w[min(pos[q] for q in op.qubits)] += 1  # its edge index
        else:  # PhasedXZ
            node_w[pos[op.qubits[0]]] += 1  # its position
    return edge_w, node_w

## 5. `create_dicke_state`

Candidate families per $k=\min(n,N-n)$:

| $k$ | Families tried |
| --- | --- |
| 0 | No entangling gates: the $N$ qubits with the best relevant readout error, optionally flipped |
| 1 | Excitation trees (all roots × 3 depth penalties) |
| $\ge2$ | BE paths (top beams) **and** two-arm split ladders (top ladders × arm extensions) |

Every candidate is compiled and scored with the predicted log-fidelity; the best one with at most 1750 moments is returned.
A wall-clock budget (30 s, leaving a 2x margin under the 60 s limit for slower machines) stops the search early. Impossible requests
($n<0$, $n>N$, more qubits than usable ones) raise `NotImplementedError`.

In [7]:
def _x_all(qubits):
    """Instructions flipping every qubit (complement trick)."""
    return [_u(q, _X) for q in qubits]  # merged into the final 1q layer by the compiler


def generate_candidates(N, n, cal, *, dd=True, deadline=None, max_trees=40, max_paths=10, max_ladders=12):
    """Yield exact candidates (circuit, qubit_list, kind) for |D^(N)_n> on the calibrated device."""
    deadline = deadline if deadline is not None else time.monotonic() + 30.0  # default budget
    k = min(n, N - n)  # prepare the lighter of the two complementary states
    flip = n > N - n  # append X everywhere afterwards if we prepared the complement
    p_exc = n / N  # excitation probability of each Dicke qubit (for readout pricing)
    if k == 0:  # computational basis state: no entanglement needed
        qs = sorted(cal.sq, key=lambda q: ((cal.e1[q] if flip else cal.e0[q]) + cal.sq[q], str(q)))[:N]  # best readout
        gate = cirq.PhasedXZGate(x_exponent=1 if flip else 0, z_exponent=0, axis_phase_exponent=0)  # X or identity
        yield cirq.Circuit(cirq.Moment(gate.on(q) for q in qs)), qs, 'trivial'  # explicit gate keeps qubits visible
        return  # only candidate
    if k == 1:  # W-state trees
        seen = set()  # avoid compiling duplicate trees
        proxies = []  # (proxy cost, root name, root, parent map)
        for w in (0.0, 0.01, 0.03):  # depth penalties: from pure quality to compact trees
            for r in sorted(cal.sq, key=str):  # every root
                parent = grow_tree(cal, r, N, p_exc, w)  # greedy tree
                if parent is None:  # component too small
                    continue  # skip
                key = (r, frozenset(parent.items()))  # identity of the tree
                if key in seen:  # duplicate
                    continue  # skip
                seen.add(key)  # remember it
                cost = (sum(2 * cal.edge_cost(v, u) for v, u in parent.items())
                        + sum(cal.ro_cost(q, p_exc) for q in [r, *parent]))  # cheap proxy: CZs + readout
                proxies.append((cost, str(r), r, parent))  # keep
        proxies.sort(key=lambda t: (t[0], t[1]))  # best proxies first
        for _, _, r, parent in proxies[:max_trees]:  # compile the most promising trees
            qs = [r, *parent]  # its qubits
            ops = w_tree_ops(r, _tree_children(parent)) + (_x_all(qs) if flip else [])  # exact construction
            yield compile_layers(ops, dd=dd), qs, 'tree'  # candidate
            if time.monotonic() > deadline:  # out of time
                return  # stop
        return  # trees beat paths for k = 1 (same CZ count, logarithmic instead of linear depth)
    edge_w, node_w = path_profile(N, k)  # per-position weights of the BE path
    for _, path in beam_paths(cal, N, edge_w, node_w, p_exc)[:max_paths]:  # best paths
        ops = dicke_path_ops(list(path), k) + (_x_all(path) if flip else [])  # exact construction
        yield compile_layers(ops, dd=dd), list(path), 'path'  # candidate
        if time.monotonic() > deadline:  # out of time
            return  # stop
    a, b = N // 2, N - N // 2  # balanced arms (b >= a >= k because k <= N/2)
    ew, nw = path_profile(b, k)  # typical arm weights
    w_edge = sum(ew) / max(len(ew), 1)  # average CZs per arm edge
    w_node = sum(nw) / max(len(nw), 1)  # average 1q gates per arm qubit
    for _, left0, right0 in ladder_candidates(cal, k, p_exc, w_edge, w_node)[:max_ladders]:  # best ladders
        used = set(left0) | set(right0)  # ladder qubits
        for _, right in extend_path(cal, right0, b, used, p_exc, w_edge, w_node)[:2]:  # right-arm options
            for _, left in extend_path(cal, left0, a, used | set(right), p_exc, w_edge, w_node)[:1]:  # best left arm
                qs = list(left) + list(right)  # all qubits
                ops = split_ops(list(left), list(right), k) + (_x_all(qs) if flip else [])  # exact construction
                yield compile_layers(ops, dd=dd), qs, 'split'  # candidate
        if time.monotonic() > deadline:  # out of time
            return  # stop


def create_dicke_state(
    num_qubits: int,
    num_excitations: int,
    cz_pauli_error_rate: dict[
        tuple[cirq.GridQubit, cirq.GridQubit], tuple[float, float]
    ],
    sq_pauli_error_rate: dict[cirq.GridQubit, tuple[float, float]],
    zero_errors: dict[cirq.GridQubit, tuple[float, float]],
    one_errors: dict[cirq.GridQubit, tuple[float, float]],
) -> tuple[cirq.Circuit, set[cirq.GridQubit]]:
    """Create a Cirq circuit that produces a Dicke state.

    Args:
        num_qubits: The number, N, of qubits in the Dicke state.
        num_excitations: The number, n, of 1s in the state.
        cz_pauli_error_rate: Measured CZ Pauli error rates and their statistical uncertainties.
        sq_pauli_error_rate: Measured single-qubit Pauli error rates and their statistical uncertainties.
        zero_errors: Measured parallel 0 -> 1 readout error rates and their statistical uncertainties.
        one_errors: Measured 1 -> 0 readout error rates and their statistical uncertainties.

    Returns:
        A Cirq circuit to produce Dicke state and a set indicating which qubits the resulting state lives on.

    Raises:
        NotImplementedError: The requested (N, n) combination is not implemented.
    """
    start = time.monotonic()  # the challenge counts > 60 s as NotImplementedError
    N, n = int(num_qubits), int(num_excitations)  # accept numpy integers too
    if N < 1 or n < 0 or n > N:  # no such Dicke state
        raise NotImplementedError(f'invalid (N, n) = ({N}, {n})')  # explicit refusal
    cal = Calibration(cz_pauli_error_rate, sq_pauli_error_rate, zero_errors, one_errors)  # pessimistic model
    if N > len(cal.sq):  # not enough working qubits
        raise NotImplementedError('not enough usable qubits')  # cannot embed
    best = None  # (predicted log-fidelity, circuit, qubits)
    for circ, qs, _ in generate_candidates(N, n, cal, deadline=start + 30.0):  # exact candidates
        if len(circ) > 1750:  # would not be run on the hardware
            continue  # discard
        score = estimate_log_fidelity(circ, cal, qs, n / N)  # predicted log success probability
        if best is None or score > best[0]:  # keep the best
            best = (score, circ, qs)  # new incumbent
    if best is None:  # nothing embeddable / everything too deep
        raise NotImplementedError('no embedding within the 1750-moment limit')  # explicit refusal
    return best[1], set(best[2])  # circuit and the qubits that hold the state

For small $N$, you may check that your circuit produces the correct state using the following function:

In [8]:
def create_dicke_statevector(num_qubits: int, num_excitations: int) -> np.ndarray:
    psi = np.zeros(2**num_qubits)
    normalization = 1 / np.sqrt(math.comb(num_qubits, num_excitations))
    qubits = range(num_qubits)
    for excited_qubits in itertools.combinations(qubits, num_excitations):
        idx = sum(2**qubit_number for qubit_number in excited_qubits)
        psi[idx] = normalization
    return psi


def test_circuit(circuit: cirq.Circuit, num_excitations: int, qubits: set[cirq.GridQubit], sampler = cirq.Simulator()):
    all_qubits = circuit.all_qubits()
    ancilla_qubits = all_qubits.difference(qubits)
    ordered_qubits = sorted(qubits) + sorted(ancilla_qubits)
    keep_indices = range(len(qubits))  # FIX (DuQuantum): range(qubits) raised TypeError on a set
    num_qubits = len(qubits)
    psi_circuit_full = sampler.simulate(circuit, qubit_order=ordered_qubits).final_state_vector
    psi_circuit = cirq.sub_state_vector(
        state_vector=psi_circuit_full,
        keep_indices=keep_indices
    ) if len(all_qubits) > len(qubits) else psi_circuit_full
    assert np.isclose(np.abs(psi_circuit @ create_dicke_statevector(num_qubits, num_excitations)) ** 2, 1.0)

### Requirements:
* The resulting circuit is allowed to contain only
  * CZ gates between pairs of qubits included in `cz_pauli_error_rate`, which will be a subset (and most) of the connected pairs in the [spec sheet](https://quantumai.google/static/site-assets/downloads/willow-spec-sheet.pdf).
  * Arbitrary single-qubit unitaries (`cirq.PhasedXZGate`) on qubits included in `sq_pauli_error_rate`, which will be most of the qubits in the spec sheet.
  * Optionally, mid-circuit reset gates.
* Classically-conditioned gates are not supported.
* In the absence of noise, the circuit must either produce perfect Dicke states of the requested $N$ and $n$ or throw a `NotImplementedError`.
* If your circuit construction code takes longer than a minute to return a circuit, that is counted the same as returning a `NotImplementedError`.
* Any circuits containing more than 1750 moments will not be run.

# Scoring

We prepare the Dicke state on actual quantum hardware and measure it in both the Z basis and the X basis. We use the following `overall_score` function to compare against the expected distributions.

In [9]:
def check_z_bitstring_distribution(
    bitstrings: np.ndarray, num_excitations: int
) -> float:
    """Measure how close the observed Z-basis distribution of bitstrings is to
    the ideal one. The lower the score, the better.

    Args:
        bitstrings: An array of shape (n_shots, n_qubits) containing the n_shots Z-basis bitstrings.
        num_excitations: The weight, n, of the Dicke state.

    Returns:
        A measure of how close the distribution is to the ideal one.
    """
    # first, let's see how many bitstrings have one excitation
    n_shots, n_qubits = bitstrings.shape
    measured_excitations = np.sum(bitstrings, axis=1)
    fraction_in_subspace = np.mean(measured_excitations == num_excitations)

    # now, let's check how uniform the distribution is over states with n excitations
    chi2 = 0
    ideal_prob = 1 / math.comb(n_qubits, num_excitations)
    for indices in itertools.combinations(range(n_qubits), num_excitations):
        ideal_state = np.zeros(n_qubits)
        ideal_state[np.array(indices)] = 1
        match_count = np.sum(np.all(bitstrings == ideal_state, axis=1))
        expected_count = n_shots * fraction_in_subspace * ideal_prob
        chi2 += (match_count - expected_count) ** 2 / expected_count

    return ideal_prob * chi2 / fraction_in_subspace


def get_correlator_average(bitstrings: np.ndarray, correlator_size: int) -> float:
    """Get the average of all weight-correlator_size correlators in `bitstrings`.

    Args:
        bitstrings: The input bitstrings, shape (n_shots, n_qubits).
        correlator_size: Compute all correlators of this weight.

    Returns:
        The average of all weight-k correlators.
    """
    correlator_sum = 0.0
    n_shots, n_qubits = bitstrings.shape
    for indices in itertools.combinations(range(n_qubits), correlator_size):
        correlator_sum += 1 - 2 * np.mean(np.sum(bitstrings[:, indices], axis=1) % 2)
    return correlator_sum / math.comb(n_qubits, correlator_size)


def check_x_bitstring_distribution(
    bitstrings: np.ndarray, num_excitations: int, max_correlator_size: int
) -> float:
    """Check expected correlations in the X basis. The lower the score, the better.

    Args:
        bitstrings: An array of shape (n_shots, n_qubits) containing the n_shots X-basis bitstrings.
        max_correlator_size: The maximum weight of correlators to consider.
        num_excitations: The weight, n, of the Dicke state.

    Returns:
        A measure of how the correlations in the X-basis measurements compare to expectation.
    """
    comb = lambda n, k: math.comb(n, k) if (k >= 0 and n >= k) else 0
    n_shots, n_qubits = bitstrings.shape
    score = 0.0
    for correlator_size in range(1, max_correlator_size + 1):
        expected_correlator = (
            comb(correlator_size, correlator_size // 2)
            * comb(n_qubits - correlator_size, num_excitations - correlator_size // 2)
            / comb(n_qubits, num_excitations)
            if correlator_size % 2 == 0
            else 0.0
        )
        measured_correlator = get_correlator_average(bitstrings, correlator_size)
        score += (
            (expected_correlator - measured_correlator) ** 2 / expected_correlator**2
            if expected_correlator != 0
            else (expected_correlator - measured_correlator) ** 2
        )

    return score


def overall_score(
    z_bitstrings: np.ndarray,
    x_bitstrings: np.ndarray,
    num_excitations: int,
    max_correlator_size: int,
) -> float:
    """An overall score that checks how the Z and X basis bitstrings compare to their ideal
    distributions. The higher the score, the better.

    Args:
        z_bitstrings: The bitstrings measured in the Z basis
        x_bitstrings: The bitstrings measured in the X basis
        num_excitations: The weight, n, of the Dicke state.
        max_correlator_size: The maximum weight of correlators to consider

    Returns:
        A measure of how similar the bitstrings are to their ideal distributions.
    """
    return 1 / (
        check_z_bitstring_distribution(z_bitstrings, num_excitations)
        + check_x_bitstring_distribution(
            x_bitstrings, num_excitations, max_correlator_size
        )
    )


### Judging criteria:
* We will run circuits generated by your code for various choices of `num_qubits` and `num_excitations` on real quantum hardware, measuring in the Z basis as well as in the X basis, each for 10,000 shots.
* For each choice of `num_qubits` and `num_excitations`, we use the `overall_score` function to score the performance. We then sum this over the choices of `num_qubits`.
* The solution with the highest score wins.

### Things to consider:
* It is permitted to use AI to answer these questions, but please include any prompts you gave to the AI in your submission.
* This will be run on real quantum hardware, so techniques like dynamical decoupling are important.


You may use the following functions to generate fake error rates to test your algorithm.

In [10]:
def pick_single_qubit_dropouts(
    dropout_rate: float,
    rng: np.random.Generator = np.random.default_rng()
) -> set[cirq.GridQubit]:
    """Pick some qubits to be broken.

    Args:
        dropout_rate: The probability that a qubit is broken.
        rng: The pseudo-random number generator

    Returns:
        The broken qubits.
    """
    all_qubits = np.array(list(cirq_google.Willow105.metadata.qubit_set))
    is_dropout = rng.random(len(all_qubits)) < dropout_rate
    return set(all_qubits[is_dropout])


def generate_qubit_error_rates(
    median: float,
    spread: float,
    dropouts: set[cirq.GridQubit],
    uncertainty: float,
    rng: np.random.Generator = np.random.default_rng(),
) -> dict[cirq.GridQubit, tuple[float, float]]:
    """Generate fake single-qubit error rates.

    Args:
        median: The median error rate.
        spread: The spread of the lognormal distribution
        dropouts: Qubits to exclude
        uncertainty: The uncertainty of the measured error rates.
        rng: The pseudo-random number generator

    Returns:
        A dictionary from qubits to the measured error rate and statistical uncertainty.
    """

    all_qubits = np.array(list(cirq_google.Willow105.metadata.qubit_set))
    is_working = np.array([q not in dropouts for q in all_qubits])
    working_qubits = all_qubits[is_working]
    error_rate = rng.lognormal(np.log(median), spread, len(working_qubits))
    return dict(
        zip(working_qubits, zip(error_rate, uncertainty * np.ones(len(working_qubits))))
    )


def generate_two_qubit_pauli_error_rates(
    median: float = 2.5e-3,
    spread: float = 0.5,
    dropout_rate: float = 0.01,
    uncertainty: float = 5e-4,
    rng: np.random.Generator = np.random.default_rng(),
    single_qubit_dropouts: set[cirq.GridQubit] = None,
) -> dict[tuple[cirq.GridQubit, cirq.GridQubit], tuple[float, float]]:
    """Generate fake two-qubit error rates.

    Args:
        median: The median error rate.
        spread: The spread of the lognormal distribution
        dropout_rate: The probability of a broken coupler.
        uncertainty: The uncertainty of the measured error rates.
        rng: The pseudo-random number generator
        single_qubit_dropouts: The single-qubit dropouts to skip.

    Returns:
        A dictionary from qubits to the measured error rate and statistical uncertainty.
    """
    if single_qubit_dropouts is None:
        single_qubit_dropouts = set()

    all_pairs = np.array(list(cirq_google.Willow105.metadata.qubit_pairs))
    no_qubit_dropouts = np.array(
        [np.all([q not in single_qubit_dropouts for q in pair]) for pair in all_pairs]
    )
    is_working = (rng.random(len(all_pairs)) > dropout_rate) * no_qubit_dropouts
    working_pairs = [tuple(pair) for pair in all_pairs[is_working]]
    error_rate = rng.lognormal(np.log(median), spread, len(working_pairs))
    return dict(
        zip(working_pairs, zip(error_rate, uncertainty * np.ones(len(working_pairs))))
    )


In [11]:
single_qubit_dropouts = pick_single_qubit_dropouts(0.01)
sq_pauli_error_rate = generate_qubit_error_rates(4e-4, 0.5, single_qubit_dropouts, 2e-4)
cz_pauli_error_rate = generate_two_qubit_pauli_error_rates(
    single_qubit_dropouts=single_qubit_dropouts
)
zero_errors = generate_qubit_error_rates(5e-3, 0.5, single_qubit_dropouts, 2e-4)
one_errors = generate_qubit_error_rates(1e-2, 0.5, single_qubit_dropouts, 3e-4)

# DuQuantum verification

Exactness is established in two layers, as the plan requires:

1. **Building blocks** are compared with their defining isometries on every input of their domain, *including the relative
   phases* (all outputs must share one global phase). This checks derivations (a)–(f) of Section 1.
2. **Complete circuits** returned by `create_dicke_state` are simulated with the notebook's own `test_circuit` for every
   $1\le N\le 10$ and every $0\le n\le N$, on two independent fake calibrations (the notebook's random one and a seeded one).
   Each circuit is also checked against the hardware rules: only `CZ` on calibrated couplers, only `PhasedXZGate` on
   calibrated qubits, homogeneous moments, at most 1750 moments, no ancillas, exactly $N$ returned qubits.

The construction is analytic, so a numerical check at small $N$ plus the algebraic argument covers all $N$: no step depends on
$N$ except through rotation angles computed in closed form.

In [12]:
def _instr_unitary(ops, qubits):
    """Exact unitary of an instruction list (tensor order = `qubits`)."""
    return compile_layers(ops, dd=True).unitary(qubit_order=qubits)  # compile WITH DD to prove DD is exact too


def _check_isometry(ops, qubits, mapping):
    """Assert that ops maps each input bitstring to its target state, all with one common global phase."""
    U = _instr_unitary(ops, qubits)  # full unitary
    ref = None  # common phase
    for bits, out in mapping:  # each domain element
        target = np.zeros(2 ** len(qubits), dtype=complex)  # target state vector
        for b, amp in out.items():  # fill amplitudes
            target[int(b, 2)] = amp  # big-endian bit order matches qubit_order
        overlap = np.vdot(target, U[:, int(bits, 2)])  # <target|U|input>
        assert abs(abs(overlap) - 1) < 1e-9, (bits, overlap)  # correct state ...
        ref = overlap if ref is None else ref  # remember the first phase
        assert abs(overlap - ref) < 1e-9, (bits, overlap, ref)  # ... with a common phase (relative phases exact)


_q = cirq.LineQubit.range(3)  # three test qubits
for _p in (0.1, 0.5, 0.9):  # several split probabilities
    _c, _s = math.sqrt(1 - _p), math.sqrt(_p)  # stay / move amplitudes
    _check_isometry(_transfer(_q[0], _q[1], _p), _q[:2], [('00', {'00': 1}), ('10', {'10': _c, '01': _s})])  # (c)
    _th = _split_angle(_p)  # rotation angle for probability p
    _check_isometry(_givens(_q[0], _q[1], _th), _q[:2], [('00', {'00': 1}), ('11', {'11': 1}),
                    ('01', {'01': _c, '10': _s}), ('10', {'10': _c, '01': -_s})])  # (d): full Givens rotation
    _check_isometry(_g3(*_q, _th), _q, [('000', {'000': 1}), ('011', {'011': 1}), ('111', {'111': 1}),
                    ('001', {'001': _c, '010': _s}), ('101', {'110': 1})])  # (e)
    _pn = min(1.0, _p + 0.05)  # a second, different probability for the top block
    _check_isometry(_top_block(*_q, _p, _pn), _q, [('000', {'000': 1}),
                    ('001', {'001': math.sqrt(1 - _pn), '010': math.sqrt(_pn)}),
                    ('011', {'011': _c, '110': _s})])  # (f)
print('building blocks (a)-(f): exact on their whole domains, including relative phases')

_grid = [cirq.GridQubit(r, c) for r in range(2) for c in range(6)]  # 2 x 6 patch for logical tests
for _N in range(2, 7):  # split construction on an explicit ladder, all k <= N/2
    for _k in range(1, _N // 2 + 1):  # every excitation number
        _right = [cirq.GridQubit(0, i) for i in range(_N - _N // 2)]  # r_1 ... r_b along row 0
        _left = [cirq.GridQubit(1, _k - v) for v in range(1, _N // 2 + 1)]  # l_v under r_{k+1-v}, continuing left
        _circ = compile_layers(split_ops(_left, _right, _k), dd=True)  # compile with DD
        test_circuit(_circ, _k, set(_left + _right))  # notebook's own checker (fixed version)
print('split construction: exact for all N <= 6')

building blocks (a)-(f): exact on their whole domains, including relative phases
split construction: exact for all N <= 6


In [13]:
import time as _time  # wall-clock timing of the full solution


def check_hardware_rules(circuit, qubits, cz_rates, sq_rates, num_qubits):
    """Assert the challenge's hardware requirements for a returned (circuit, qubits) pair."""
    pairs = {frozenset(p) for p in cz_rates}  # calibrated couplers
    assert len(qubits) == num_qubits, 'wrong number of qubits'  # exactly N data qubits
    assert circuit.all_qubits() <= set(qubits), 'unexpected ancilla'  # no extra qubits
    assert len(circuit) <= 1750, 'too many moments'  # hardware limit
    for moment in circuit:  # every moment
        assert len({len(op.qubits) for op in moment}) == 1, 'mixed moment'  # 1q-only or CZ-only
        for op in moment:  # every gate
            if len(op.qubits) == 2:  # two-qubit gate
                assert op.gate == cirq.CZ and frozenset(op.qubits) in pairs, op  # CZ on a calibrated coupler
            else:  # one-qubit gate
                assert isinstance(op.gate, cirq.PhasedXZGate) and op.qubits[0] in sq_rates, op  # native gate


def _seeded_calibration(seed):
    """A reproducible fake calibration using the notebook's generators."""
    rng = np.random.default_rng(seed)  # seeded generator
    drop = pick_single_qubit_dropouts(0.01, rng)  # broken qubits
    sq = generate_qubit_error_rates(4e-4, 0.5, drop, 2e-4, rng)  # single-qubit errors
    cz = generate_two_qubit_pauli_error_rates(single_qubit_dropouts=drop, rng=rng)  # CZ errors
    e0 = generate_qubit_error_rates(5e-3, 0.5, drop, 2e-4, rng)  # 0 -> 1 readout
    e1 = generate_qubit_error_rates(1e-2, 0.5, drop, 3e-4, rng)  # 1 -> 0 readout
    return cz, sq, e0, e1


_cals = {'notebook calibration': (cz_pauli_error_rate, sq_pauli_error_rate, zero_errors, one_errors),
         'seeded calibration': _seeded_calibration(7)}  # two independent devices
_slowest = 0.0  # longest construction time
for _name, _cal in _cals.items():  # both calibrations
    for _N in range(1, 11):  # all small sizes
        for _n in range(0, _N + 1):  # all excitation numbers, including n > N/2 and the trivial ones
            _t = _time.monotonic()  # start timer
            _circ, _qs = create_dicke_state(_N, _n, *_cal)  # the submission function
            _slowest = max(_slowest, _time.monotonic() - _t)  # track the worst case
            check_hardware_rules(_circ, _qs, _cal[0], _cal[1], _N)  # hardware rules
            test_circuit(_circ, _n, _qs)  # exact state (notebook's checker)
    print(f'{_name}: all (N, n) with N <= 10 exact and hardware-valid')
print(f'slowest construction: {_slowest:.1f} s (limit 60 s)')

notebook calibration: all (N, n) with N <= 10 exact and hardware-valid


seeded calibration: all (N, n) with N <= 10 exact and hardware-valid
slowest construction: 0.8 s (limit 60 s)


Resource summary on the notebook's calibration (CZ count, moments, construction time), including larger instances that
are too big to simulate as state vectors here but use exactly the same, already verified, code paths.

In [14]:
print(f"{'N':>3} {'n':>3} {'CZs':>5} {'moments':>8} {'time(s)':>8}")
for _N, _n in [(4, 1), (4, 2), (6, 1), (6, 3), (8, 1), (8, 2), (8, 4), (10, 5), (12, 1), (12, 6), (16, 2), (20, 1), (20, 10), (40, 1), (100, 1)]:
    _t = _time.monotonic()  # start timer
    _circ, _qs = create_dicke_state(_N, _n, cz_pauli_error_rate, sq_pauli_error_rate, zero_errors, one_errors)  # build
    _dt = _time.monotonic() - _t  # elapsed
    check_hardware_rules(_circ, _qs, cz_pauli_error_rate, sq_pauli_error_rate, _N)  # rules still hold
    _ncz = sum(1 for op in _circ.all_operations() if len(op.qubits) == 2)  # CZ count
    print(f'{_N:>3} {_n:>3} {_ncz:>5} {len(_circ):>8} {_dt:>8.1f}')

  N   n   CZs  moments  time(s)
  4   1     5        7      0.0
  4   2     7        9      0.1
  6   1     9       11      0.1


  6   3    23       25      0.2
  8   1    13       15      0.1


  8   2    23       23      0.2


  8   4    49       49      0.4


 10   5    85       79      0.7


...

 16   2    55       47      0.7


 20   1    37       27      0.4


 20  10   415      229      3.4


 40   1    77       47      1.0


100   1   197       55      2.2


# DuQuantum noisy emulator and design experiments

Hardware access is not available, so design choices were compared on a density-matrix emulator whose assumptions are
explicit. Per moment it applies:

* depolarising noise after every gate, with the calibration's Pauli error rate ($p/3$ per Pauli for 1q gates, $p/15$ for CZ);
* amplitude damping ($T_1=68\,\mu$s) and white dephasing ($T_\phi=100\,\mu$s) on every qubit for the moment's duration;
* optional **coherent** errors that dynamical decoupling targets: quasi-static detuning (a random frequency offset per qubit,
  averaged over several realisations) and *spectator phases* (a fixed random $Z$ rotation on an idle qubit whenever a
  neighbour takes part in a CZ);
* independent asymmetric readout flips using the calibrated $0\to1$ and $1\to0$ rates.

**Shot-noise-free score.** With 10,000 shots the official $L_Z$ fluctuates by about ±30%, which would swamp design differences.
Writing $S$ for the shots, $f$ for the fraction of weight-$k$ shots, $M=\binom Nk$ and $q_i$ for the conditional distribution over
the $M$ weight-$k$ strings, the notebook's $L_Z$ equals $S\big[\sum_i (c_i/m)^2-\frac1M\big]$ with $m=Sf$, so for multinomial counts
$$\mathbb E[L_Z]=S\Big(\sum_iq_i^2-\frac1M\Big)+\frac{1-\sum_iq_i^2}{f}.$$
The first term penalises non-uniformity (heavily, because it is multiplied by $S$), the second is $\approx1/f$: even a perfect
state scores $\mathbb E[L_Z]=(1-1/M)$, so the ideal expected score is $M/(M-1)$. $L_X$ is evaluated at its exact (infinite-shot)
correlators. Comparisons below use this expected score on identical calibrations.

In [15]:
EMU_T1 = 68e-6  # amplitude-damping time (s), typical published Willow value
EMU_TPHI = 100e-6  # white pure-dephasing time (s), an assumption
EMU_T_1Q, EMU_T_CZ = 25e-9, 42e-9  # moment durations (s)


class HardwareNoise:
    """Per-moment noise channels for the emulator (see the list above)."""

    def __init__(self, cal, detuning_hz=None, spectator=None):
        cz, sq = cal[0], cal[1]  # calibration dictionaries (means are used as the 'true' rates)
        self.cz_err = {frozenset(k): v[0] for k, v in cz.items()}  # CZ Pauli error per coupler
        self.sq_err = {q: v[0] for q, v in sq.items()}  # single-qubit Pauli error per qubit
        self.detuning_hz = detuning_hz or {}  # qubit -> quasi-static frequency offset (Hz)
        self.spectator = spectator or {}  # (idle qubit, active neighbour) -> Z phase per CZ moment
        self.nbrs = {}  # device adjacency (for spectator phases)
        for pair in self.cz_err:  # every coupler
            a, b = tuple(pair)  # endpoints
            self.nbrs.setdefault(a, set()).add(b)  # undirected ...
            self.nbrs.setdefault(b, set()).add(a)  # ... adjacency

    def moment_ops(self, moment, qubits):
        """Noise operations that follow `moment` on the simulated `qubits`."""
        dur = EMU_T_CZ if any(len(op.qubits) == 2 for op in moment) else EMU_T_1Q  # moment duration
        out = []  # noise operations
        for op in moment:  # gate errors
            if len(op.qubits) == 2:  # CZ
                out.append(cirq.depolarize(self.cz_err[frozenset(op.qubits)], n_qubits=2).on(*op.qubits))  # 2q Pauli noise
            else:  # PhasedXZ
                out.append(cirq.depolarize(self.sq_err[op.qubits[0]]).on(*op.qubits))  # 1q Pauli noise
        if self.spectator and dur == EMU_T_CZ:  # coherent spectator phases during CZ moments
            active = {q for op in moment for q in op.qubits}  # qubits inside CZs
            for q in qubits:  # idle qubits only
                if q not in active:  # spectator
                    phi = sum(self.spectator.get((q, w), 0.0) for w in self.nbrs.get(q, ()) if w in active)  # total phase
                    if phi:  # non-zero
                        out.append(cirq.rz(phi).on(q))  # coherent Z error
        g = 1 - math.exp(-dur / EMU_T1)  # amplitude-damping probability
        lam = 1 - math.exp(-2 * dur / EMU_TPHI)  # phase-damping parameter
        for q in qubits:  # idling noise on every qubit
            out.append(cirq.amplitude_damp(g).on(q))  # T1
            out.append(cirq.phase_damp(lam).on(q))  # T_phi
            if self.detuning_hz.get(q):  # quasi-static detuning
                out.append(cirq.rz(2 * math.pi * self.detuning_hz[q] * dur).on(q))  # coherent Z drift
        return out


def noisy_density_matrix(circuit, noise, qubits):
    """Final density matrix of `circuit` with noise inserted after every moment."""
    noisy = cirq.Circuit()  # noisy circuit
    for moment in circuit:  # original moments
        noisy.append(cirq.Moment(moment.operations))  # the gates
        extra = noise.moment_ops(moment, qubits)  # their noise
        while extra:  # pack noise into non-overlapping moments
            used, this, rest = set(), [], []  # greedy packing
            for op in extra:  # each channel
                if used.isdisjoint(op.qubits):  # fits into this noise moment
                    this.append(op)  # take it
                    used.update(op.qubits)  # occupy its qubits
                else:  # overlaps an earlier channel
                    rest.append(op)  # next noise moment
            noisy.append(cirq.Moment(this))  # one noise moment
            extra = rest  # remaining channels
    return cirq.DensityMatrixSimulator(dtype=np.complex128).simulate(noisy, qubit_order=qubits).final_density_matrix


def _readout_channel(probs, n, e0, e1):
    """Exact outcome distribution after independent asymmetric bit flips (qubit 0 = most significant bit)."""
    p = np.asarray(probs, dtype=float).reshape([2] * n)  # one axis per qubit
    for q in range(n):  # flip each qubit independently
        m = np.array([[1 - e0[q], e1[q]], [e0[q], 1 - e1[q]]])  # m[observed, true]
        p = np.moveaxis(np.tensordot(m, p, axes=([1], [q])), 0, q)  # apply along axis q
    return p.reshape(-1)  # back to a vector


def expected_score(pz, px, n_qubits, k, e0, e1, shots=10_000, max_corr=4):
    """Shot-noise-free expectation of the official losses (formula above)."""
    pz = _readout_channel(pz, n_qubits, e0, e1)  # observed Z-basis distribution
    px = _readout_channel(px, n_qubits, e0, e1)  # observed X-basis distribution
    weights = np.array([bin(i).count('1') for i in range(2 ** n_qubits)])  # Hamming weights
    f = pz[weights == k].sum()  # fraction in the weight-k subspace
    q = pz[weights == k] / f  # conditional distribution
    M = math.comb(n_qubits, k)  # subspace size
    lz = shots * (np.sum(q ** 2) - 1 / M) + (1 - np.sum(q ** 2)) / f  # E[L_Z]
    lx = 0.0  # exact L_X
    comb = lambda a, b: math.comb(a, b) if (b >= 0 and a >= b) else 0  # binomial with zero outside range
    for c in range(1, max_corr + 1):  # correlator orders
        meas = 0.0  # average c-body correlator
        for idx in itertools.combinations(range(n_qubits), c):  # all subsets
            parity = np.zeros(2 ** n_qubits, dtype=int)  # parity of the subset per basis state
            for qb in idx:  # accumulate parity
                parity ^= (np.arange(2 ** n_qubits) >> (n_qubits - 1 - qb)) & 1  # bit of qubit qb
            meas += np.sum(px * (1 - 2 * parity))  # <prod X>
        meas /= math.comb(n_qubits, c)  # average
        expv = comb(c, c // 2) * comb(n_qubits - c, k - c // 2) / comb(n_qubits, k) if c % 2 == 0 else 0.0  # ideal value
        lx += (expv - meas) ** 2 / expv ** 2 if expv != 0 else (expv - meas) ** 2  # official weighting
    return float(1 / (lz + lx))  # expected official score


def emulate(circuit, qubits, k, cal, *, detuning_sigma_hz=0.0, n_samples=1, spectator_sigma=0.0, seed=1):
    """Expected official score of a returned circuit under the emulator."""
    rng = np.random.default_rng(seed)  # reproducible noise realisations
    qubits = sorted(qubits)  # measurement order
    n_q = len(qubits)  # number of data qubits
    spec = {}  # spectator phases
    if spectator_sigma:  # draw one fixed device realisation
        for pair in sorted(cal[0], key=str):  # every coupler
            a, b = tuple(pair)  # endpoints
            spec[(a, b)] = rng.normal(0, spectator_sigma)  # phase on a when b is active
            spec[(b, a)] = rng.normal(0, spectator_sigma)  # phase on b when a is active
    H = cirq.unitary(cirq.Circuit(cirq.H(q) for q in qubits))  # X-basis measurement rotation
    pz = np.zeros(2 ** n_q)  # Z-basis distribution
    px = np.zeros(2 ** n_q)  # X-basis distribution
    for _ in range(n_samples):  # average over quasi-static realisations
        det = {q: rng.normal(0, detuning_sigma_hz) for q in qubits} if detuning_sigma_hz else {}  # detunings
        rho = noisy_density_matrix(circuit, HardwareNoise(cal, det, spec), qubits)  # noisy state
        pz += np.real(np.diag(rho)) / n_samples  # Z-basis probabilities
        px += np.real(np.diag(H @ rho @ H.conj().T)) / n_samples  # X-basis probabilities
    e0 = np.array([cal[2][q][0] for q in qubits])  # readout 0 -> 1
    e1 = np.array([cal[3][q][0] for q in qubits])  # readout 1 -> 0
    return expected_score(pz, px, n_q, k, e0, e1)  # shot-noise-free official score

### Design experiments

The full runs (`tools/experiments.py` in the repository; two or three calibration seeds each) gave the following
expected official scores (ideal values: $M/(M-1)$, e.g. 1.05 for $(6,3)$).

**A. Construction family** (same calibration, best placement of each family):

| $(N,n)$ | BE path: CZs / moments / score | Two-arm split: CZs / moments / score |
| --- | --- | --- |
| (6,2) | 18 / 31 / 0.878 | **15 / 17 / 0.943** |
| (6,3) | 36 / 63 / 0.733 | **23 / 25 / 0.859** |
| (8,2) | 26 / 43 / 0.794 | **23 / 23 / 0.876** |
| (8,3) | 54 / 91 / 0.575 | **41 / 41 / 0.753** |
| (8,4) | 77 / 117 / 0.502 | **49 / 49 / 0.699** |

The split construction wins everywhere: fewer CZs *and* about half the depth.

**B. Dynamical decoupling** (expected score without → with DD):

| noise model | (6,1) | (6,3) | (8,2) |
| --- | --- | --- | --- |
| incoherent only | 1.117 → 1.118 | 0.868 → 0.859 | 0.885 → 0.876 |
| + detuning, σ = 50 kHz | 1.113 → 1.114 | 0.847 → 0.848 | 0.880 → 0.873 |
| + detuning, σ = 200 kHz | 1.036 → 1.041 | **0.288 → 0.579** | **0.644 → 0.812** |
| + spectator phases, σ = 0.1 rad | 1.116 → 1.118 | **0.778 → 0.827** | **0.668 → 0.691** |

DD is an insurance policy: when idle errors are purely incoherent it costs about 1% (the pulses themselves have errors),
but when coherent idle errors are present, which is what the organisers warn about, it recovers large losses (×2 at
200 kHz detuning). We therefore keep DD on.

**C. Placement strategy**, scored on a "true" device whose rates are the reported means plus Gaussian noise of the reported
uncertainty (so the calibration is itself an estimate). Mean over 3 seeds × {(6,1), (6,3), (8,2)}:

| strategy | mean expected score |
| --- | --- |
| calibration-aware, $\beta=1$ (submitted) | **0.944** |
| calibration-aware, $\beta=0$ | 0.944 (identical choices: the fake data have uniform uncertainties) |
| random choice among our candidates | 0.918 |
| topology-only (ignores calibration) | 0.885 |

Calibration-aware placement is clearly better than ignoring calibration, consistent with Huo et al.'s observation that
randomised alternatives are competitive but not superior. The $\beta$ penalty only matters when uncertainties differ between
qubits, as they will on real hardware.

The cell below re-runs a light version of each experiment (about two minutes).

In [16]:
def _select(N, n, cal_obj, dd=True, kinds=None, rng=None):
    """Best (or random) candidate of the requested families under a given Calibration object."""
    cands = [(estimate_log_fidelity(c, cal_obj, qs, n / N), c, qs, kind)
             for c, qs, kind in generate_candidates(N, n, cal_obj, dd=dd) if not kinds or kind in kinds]  # scored
    return cands[rng.integers(len(cands))] if rng is not None else max(cands, key=lambda t: t[0])  # pick


_cal = _seeded_calibration(0)  # one reproducible device
_cobj = Calibration(*_cal)  # its pessimistic model
for _kind in ('path', 'split'):  # A. construction family on (6, 3)
    _, _c, _qs, _ = _select(6, 3, _cobj, kinds=[_kind])  # best candidate of this family
    print(f'(6,3) {_kind:5s}: CZs={sum(len(op.qubits) == 2 for op in _c.all_operations())}, '
          f'moments={len(_c)}, expected score={emulate(_c, _qs, 3, _cal):.3f}')
for _dd in (False, True):  # B. DD under strong quasi-static detuning on (6, 3)
    _, _c, _qs, _ = _select(6, 3, _cobj, dd=_dd)  # best candidate with / without DD
    print(f'(6,3) DD={_dd!s:5s}: incoherent={emulate(_c, _qs, 3, _cal):.3f}, '
          f'200 kHz detuning={emulate(_c, _qs, 3, _cal, detuning_sigma_hz=200e3, n_samples=6):.3f}')
_uniform = tuple({key: (1e-3, 0.0) for key in d} for d in _cal)  # C. topology-only calibration
for _name, _co in (('calibration-aware', _cobj), ('topology-only', Calibration(*_uniform, beta=0.0))):  # (6, 1)
    _, _c, _qs, _ = _select(6, 1, _co)  # chosen W-state circuit
    print(f'(6,1) {_name}: expected score={emulate(_c, _qs, 1, _cal):.3f}')

(6,3) path : CZs=36, moments=63, expected score=0.733


(6,3) split: CZs=23, moments=25, expected score=0.859


(6,3) DD=False: incoherent=0.868, 200 kHz detuning=0.288


(6,3) DD=True : incoherent=0.859, 200 kHz detuning=0.579
(6,1) calibration-aware: expected score=1.118


(6,1) topology-only: expected score=0.930


# Bonus question:
Now suppose we allow for adaptive circuits, i.e. those with mid-circuit measurements and subsequent gates conditioned on the outcomes of those measurements. See if you can construct shallower circuits to prepare Dicke states given this capability. We will not run these circuits on the hardware.

For all $(N, n)$, with $2 \leq N \leq 15$, and $1 \leq n \leq \lfloor N/2 \rfloor$, we will compare the depth of the circuits produced by the various submissions and assign a point to the one with the shallowest depth, measured by the number of moments in the circuit.

Specifically, your task is to complete the following function:

## 6. Bonus: shallowest circuits (adaptive circuits allowed, no noise)

The bonus scores depth only (`len(circuit)`), on the ideal Willow105 connectivity. We considered the adaptive designs
in the plan (measurement-based fan-out and the "binary index → one-hot → $X$-basis measurement → $Z$ correction" W-state
scheme). For $N\le20$ they lose to our unitary constructions: a measurement moment plus a classically controlled
correction moment already cost 2 moments, the constant-depth fan-out gadget needs several more, and the binary→one-hot
decoder needs multi-controlled gates routed across the grid, whereas the excitation tree reaches $N$ qubits in
$\approx4\lceil\log_2N\rceil+1$ moments. Moreover mid-circuit measurements must cover whole readout diagonals
($\text{row}+\text{col}=$ const), which forces ancillas onto separate diagonals and lengthens routing. We therefore submit
**non-adaptive** circuits (explicitly allowed) and spend the effort on depth:

* **W states:** a *broadcast tree*. In round $t$ every informed qubit hands the excitation to one uninformed neighbour, so the
  informed set can double every two CZ layers; we build the tree by greedy bipartite matching round by round from every
  possible root and keep the shallowest compiled circuit.
* **$k\ge2$:** the two-arm split construction (both arms in parallel) and the BE path, compiled on the ideal device, keeping
  the shallower.
* **Packing:** for the bonus, single-qubit gates may share moments with CZs on other qubits (no hardware calibration constraint
  applies), so we re-insert every operation at the earliest possible moment and keep whichever version is shorter.

In [17]:
def _ideal_calibration():
    """Uniform 'calibration' covering every Willow105 qubit and coupler (the bonus is noise-free)."""
    device = cirq_google.Willow105  # the target device
    qubits = sorted(device.metadata.qubit_set)  # all 105 qubits
    cz = {tuple(sorted(p)): (1e-3, 0.0) for p in device.metadata.qubit_pairs}  # every coupler, equal cost
    flat = {q: (1e-3, 0.0) for q in qubits}  # equal single-qubit and readout costs
    return Calibration(cz, flat, flat, flat, beta=0.0)  # deterministic, uniform model


def broadcast_tree(cal, root, N):
    """Greedy round-by-round matching tree: every informed node recruits at most one new neighbour per round."""
    informed = [root]  # informed nodes in order of recruitment
    parent = {}  # child -> parent
    center = (np.mean([q.row for q in cal.sq]), np.mean([q.col for q in cal.sq]))  # prefer central growth
    while len(informed) < N:  # until N qubits hold the excitation
        taken = set()  # qubits recruited this round
        new = []  # (child, parent) pairs of this round
        for u in list(informed):  # each informed node may recruit once
            options = [v for v in cal.adj[u] if v not in parent and v != root and v not in taken]  # free neighbours
            if not options or len(informed) + len(new) >= N:  # nothing to do / enough nodes
                continue  # next node
            v = min(options, key=lambda q: (len([w for w in cal.adj[q] if w not in parent and w != root]) == 0,
                                            (q.row - center[0]) ** 2 + (q.col - center[1]) ** 2, str(q)))  # central, non-dead-end
            taken.add(v)  # reserve it
            new.append((v, u))  # recruit
        if not new:  # stuck (cannot happen on a connected device with N <= 105)
            return None  # give up on this root
        for v, u in new:  # commit the round
            parent[v] = u  # tree edge
            informed.append(v)  # v can recruit from the next round on
    return parent  # {child: parent}


def pack_earliest(circuit):
    """Re-insert every operation at the earliest moment allowed by its qubits (mixed 1q/CZ moments)."""
    return cirq.Circuit(circuit.all_operations(), strategy=cirq.InsertStrategy.EARLIEST)  # cirq does the packing


def create_dicke_state_adaptive(
    num_qubits: int, num_excitations: int
) -> tuple[cirq.Circuit, set[cirq.GridQubit]]:
    """Create a Cirq circuit that produces a Dicke state, allowing for adaptive circuits.

    Args:
        num_qubits: The number, N, of qubits in the Dicke state.
        num_excitations: The number, n, of 1s in the state.

    Returns:
        A Cirq circuit to produce Dicke state and a set indicating which qubits the resulting state lives on.

    Raises:
        NotImplementedError: The requested (N, n) combination is not implemented.
    """
    N, n = int(num_qubits), int(num_excitations)  # accept numpy integers
    if N < 1 or n < 0 or n > N or N > 105:  # impossible request
        raise NotImplementedError(f'invalid (N, n) = ({N}, {n})')  # explicit refusal
    cal = _ideal_calibration()  # all qubits, uniform costs
    k = min(n, N - n)  # complement trick, as in the hardware solution
    flip = n > N - n  # whether to flip everything at the end
    cands = []  # (circuit, qubits)
    if k == 1:  # W states: broadcast trees from every root
        for r in sorted(cal.sq, key=str):  # every possible root
            parent = broadcast_tree(cal, r, N)  # shallow tree
            if parent is None:  # failed
                continue  # skip
            qs = [r, *parent]  # its qubits
            ops = w_tree_ops(r, _tree_children(parent)) + (_x_all(qs) if flip else [])  # exact construction
            cands.append((compile_layers(ops, dd=False), qs))  # no DD in a noise-free setting
    else:  # k = 0 or k >= 2: reuse the hardware candidate generator on the ideal device
        for circ, qs, _ in generate_candidates(N, n, cal, dd=False, max_trees=5, max_paths=2, max_ladders=3):  # few suffice
            cands.append((circ, qs))  # keep
    if not cands:  # nothing found
        raise NotImplementedError('no embedding found')  # explicit refusal
    best = None  # (depth, circuit, qubits)
    for circ, qs in cands:  # pick the shallowest, trying both layouts
        for c in (circ, pack_earliest(circ)):  # layered and packed versions
            if best is None or len(c) < best[0]:  # strictly shallower
                best = (len(c), c, qs)  # new incumbent
    return best[1], set(best[2])  # circuit and data qubits

## Scoring

We will use the following function to score submissions:

The organisers' `score_solution` calls a `validate_circuit` helper that the notebook does not define. Our local stand-in
checks the device rules with `cirq_google.Willow105.validate_circuit` and, for $N\le14$ (state-vector size $2^N$), the exact
state with the notebook's `test_circuit`. Our bonus circuits contain no measurements, so the readout-diagonal rule for
mid-circuit measurements is satisfied trivially.

In [18]:
def validate_circuit(circuit, num_excitations, qubits):
    """Local stand-in for the organisers' checker."""
    cirq_google.Willow105.validate_circuit(circuit)  # gate set and nearest-neighbour CZs on Willow105
    assert not any(cirq.is_measurement(op) for op in circuit.all_operations()), 'unexpected measurement'  # readout rule
    assert circuit.all_qubits() <= set(qubits), 'unexpected ancilla'  # state lives on the returned qubits
    if len(qubits) <= 14:  # state-vector check where affordable
        test_circuit(circuit, num_excitations, qubits)  # exact Dicke state

In [19]:
def score_solution() -> float:
    score = 0
    for num_qubits in range(2, 21):
        for num_excitations in range(1, num_qubits//2+1):
            try:
                circuit, qubits = create_dicke_state_adaptive(num_qubits, num_excitations)
            except NotImplementedError:
                continue
            validate_circuit(circuit, num_excitations, qubits) # checks that the circuit is valid
            score += num_qubits * num_excitations / len(circuit)
    return score

Running the organisers' scoring function (it loops over $2\le N\le20$, $1\le n\le\lfloor N/2\rfloor$, about a minute). For reference,
compiling the hardware-track candidates on the ideal device without the bonus-specific tree and packing gives 39.7.

In [20]:
_t = _time.monotonic()  # timer
print(f'bonus score = {score_solution():.2f}  ({_time.monotonic() - _t:.0f} s)')  # organisers' metric
for _N in (8, 12, 16, 20):  # depth table for a few sizes
    print(_N, [len(create_dicke_state_adaptive(_N, _n)[0]) for _n in range(1, _N // 2 + 1)])  # moments per n

bonus score = 65.32  (23 s)


8 [11, 22, 40, 49]


12 [14, 34, 68, 88, 100, 109]


16 [15, 46, 96, 124, 136, 148, 160, 169]


20 [18, 58, 124, 160, 172, 184, 196, 208, 220, 229]


Notes and constraints:
* We will verify classically that the circuits produce the desired Dicke state up to a global phase.
* Using adaptive circuits is not required, so you may reuse your solution from the previous question.
* Although we do not provide error rates for this question, it is still necessary to respect the connectivity of the [Willow device](https://quantumai.google/static/site-assets/downloads/willow-spec-sheet.pdf), i.e. you can only use qubits on that device and CZ gates between adjacent qubits on the device. You may use `cirq_google.Willow105.validate_circuit(circuit)` to validate that your circuits are correctly routed to such a device.
* On real hardware, qubits along a diagonal such that row # + column # = const share a readout line and measuring one qubit dephases the other qubits on that readout line. Therefore, mid-circuit measurements of some but not all of the qubits on a readout line are not allowed. (This constraint is not checked by `cirq_google.Willow105.validate_circuit(circuit)`.)
* As before, the code must execute in under a minute for each $(N, n)$.

# AI usage disclosure

The challenge allows AI assistance provided the prompts are included. This solution was designed and implemented with
Claude (Anthropic) running in Claude Code, from the following prompt:

> using @DuQuantum_Research_and_Implementation_Plan.md , design and implement a solution to @"[standard_track] Google Quantum AI Challenge.ipynb" . Make sure to document each line of code and justify design decisions, as well as explain the mathematical reasoning behind each choice.

The referenced research plan (`DuQuantum_Research_and_Implementation_Plan.md`) was itself exported from an earlier AI
conversation; the team should append the prompts of that conversation here before submitting.

Supporting research tools are kept in the repository's `tools/` directory: numerical minimum-CZ searches used to discover
and confirm the gate decompositions, the noisy emulator and design experiments, fake-calibration helpers and the script
that assembles this notebook.